In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Working directory:", Path.cwd())

Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
Pandas: 3.0.5
NumPy: 2.5.2
Scikit-learn: 1.9.0
Working directory: c:\Users\Precision5540\Desktop\G7_Retail_Analytics_Final_Project\notebooks


In [2]:
# Define project and data paths

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)

for file in DATA_RAW.glob("*.csv"):
    print("-", file.name)

Project root: c:\Users\Precision5540\Desktop\G7_Retail_Analytics_Final_Project
Raw data folder: c:\Users\Precision5540\Desktop\G7_Retail_Analytics_Final_Project\data\raw
- Campaign Response.csv
- Online Retail Customer Feedback.csv
- Online Retail Sales Data.csv


In [3]:
# Load Phase 1 datasets

sales_path = DATA_RAW / "Online Retail Sales Data.csv"
campaign_path = DATA_RAW / "Campaign Response.csv"

sales_df = pd.read_csv(sales_path)
campaign_df = pd.read_csv(campaign_path)

print("Sales dataset shape:", sales_df.shape)
print("Campaign dataset shape:", campaign_df.shape)

Sales dataset shape: (337321, 8)
Campaign dataset shape: (3834, 6)


In [4]:
print("SALES DATASET")
print(sales_df.dtypes)
display(sales_df.head())

print("\nCAMPAIGN DATASET")
print(campaign_df.dtypes)
display(campaign_df.head())

SALES DATASET
CustomerID       int64
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
Country            str
dtype: object


,CustomerID,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
0,13313,539993,22386,JUMBO BAG PINK POLKADOT,10,04/01/23 10:00,1.95,United Kingdom
1,13313,539993,21499,BLUE POLKADOT WRAP,25,04/01/23 10:00,0.42,United Kingdom
2,13313,539993,21498,RED RETROSPOT WRAP,25,04/01/23 10:00,0.42,United Kingdom
3,13313,539993,22379,RECYCLING BAG RETROSPOT,5,04/01/23 10:00,2.10,United Kingdom
4,13313,539993,20718,RED RETROSPOT SHOPPER BAG,10,04/01/23 10:00,1.25,United Kingdom



CAMPAIGN DATASET
CustomerID          int64
response            int64
n_comp              int64
loyalty             int64
nps                   str
n_communications    int64
dtype: object


,CustomerID,response,n_comp,loyalty,nps,n_communications
0,12346,1,2,1,7,8
1,12747,0,1,1,3,3
2,12748,1,0,1,9,6
3,12749,0,4,1,2,5
4,12820,0,4,1,2,2


In [5]:
print("SALES DATASET")
print("\nMissing values:")
print(sales_df.isna().sum())

print("\nDuplicate rows:")
print(sales_df.duplicated().sum())


print("\nCAMPAIGN DATASET")
print("\nMissing values:")
print(campaign_df.isna().sum())

print("\nDuplicate rows:")
print(campaign_df.duplicated().sum())

SALES DATASET

Missing values:
CustomerID     0
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

Duplicate rows:
4657

CAMPAIGN DATASET

Missing values:
CustomerID          0
response            0
n_comp              0
loyalty             0
nps                 0
n_communications    0
dtype: int64

Duplicate rows:
0


In [ ]:
print("SALES DATASET")

print("Negative quantities:", (sales_df["Quantity"] < 0).sum())
print("Zero quantities:", (sales_df["Quantity"] == 0).sum())

print("Negative unit prices:", (sales_df["UnitPrice"] < 0).sum())
print("Zero unit prices:", (sales_df["UnitPrice"] == 0).sum())

cancelled_invoices = sales_df["InvoiceNo"].astype(str).str.startswith("C")
print("Rows with cancelled invoices:", cancelled_invoices.sum())


print("\nCAMPAIGN DATASET")

nps_numeric_check = pd.to_numeric(
    campaign_df["nps"],
    errors="coerce"
)

print("NPS values that cannot be converted to numeric:")
display(
    campaign_df.loc[
        nps_numeric_check.isna(),
        ["CustomerID", "nps"]
    ]
)

print("Response values:", sorted(campaign_df["response"].unique()))
print("Loyalty values:", sorted(campaign_df["loyalty"].unique()))
print("NPS unique values:", campaign_df["nps"].unique())

SALES DATASET
Negative quantities: 6939
Zero quantities: 0
Negative unit prices: 0
Zero unit prices: 23
Rows with cancelled invoices: 6939

CAMPAIGN DATASET
NPS values that cannot be converted to numeric:


,CustomerID,nps
7,12823,
26,12844,
787,13952,


Response values: [np.int64(0), np.int64(1)]
Loyalty values: [np.int64(0), np.int64(1)]
NPS unique values: <StringArray>
['7', '3', '9', '2', '5', '6', ' ', '8', '0', '4', '1', '10']
Length: 12, dtype: str


In [7]:
negative_quantity_rows = sales_df["Quantity"] < 0

print(
    "All negative quantities are cancelled invoices:",
    (
        negative_quantity_rows
        == cancelled_invoices
    ).all()
)

print("\nSample cancelled transactions:")
display(
    sales_df.loc[
        cancelled_invoices
    ].head(10)
)

print("\nTransactions with UnitPrice = 0:")
display(
    sales_df.loc[
        sales_df["UnitPrice"] == 0
    ]
)

All negative quantities are cancelled invoices: True

Sample cancelled transactions:


,CustomerID,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
69,14606,C540006,21306,SET/4 DAISY MIRROR MAGNETS,-1,04/01/23 10:48,2.10,United Kingdom
70,14606,C540006,84352,SILVER CHRISTMAS TREE BAUBLE STAND,-1,04/01/23 10:48,16.95,United Kingdom
71,14606,C540006,22423,REGENCY CAKESTAND 3 TIER,-1,04/01/23 10:48,12.75,United Kingdom
72,15379,C540007,21055,TOOL BOX SOFT TOY,-6,04/01/23 11:08,8.95,United Kingdom
73,15379,C540007,22274,FELTCRAFT DOLL EMILY,-6,04/01/23 11:08,2.95,United Kingdom
427,16029,C540030,22070,SMALL RED RETROSPOT MUG IN BOX,-24,04/01/23 13:47,3.75,United Kingdom
641,15373,C540097,22835,HOT WATER BOTTLE I AM SO POORLY,-4,04/01/23 15:46,4.65,United Kingdom
642,15373,C540097,22423,REGENCY CAKESTAND 3 TIER,-6,04/01/23 15:46,12.75,United Kingdom
643,15373,C540097,22179,SET 10 LIGHTS NIGHT OWL,-4,04/01/23 15:46,6.75,United Kingdom
644,15373,C540097,22113,GREY HEART HOT WATER BOTTLE,-4,04/01/23 15:46,3.75,United Kingdom



Transactions with UnitPrice = 0:


,CustomerID,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
2806,13081,540372,22090,PAPER BUNTING RETROSPOT,24,06/01/23 16:41,0.0,United Kingdom
2808,13081,540372,22553,PLASTERS IN TIN SKULLS,24,06/01/23 16:41,0.0,United Kingdom
7553,15107,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,13/01/23 15:10,0.0,United Kingdom
24454,17560,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,10/02/23 13:08,0.0,United Kingdom
53342,13239,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,23/03/23 10:25,0.0,United Kingdom
59818,13113,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,30/03/23 12:45,0.0,United Kingdom
64243,14410,548871,22162,HEART GARLAND RUSTIC PADDED,2,04/04/23 14:42,0.0,United Kingdom
93234,17667,553000,47566,PARTY BUNTING,4,12/05/23 15:21,0.0,United Kingdom
152806,16818,561284,22167,OVAL WALL MIRROR DIAMANTE,1,26/07/23 12:24,0.0,United Kingdom
157170,15581,561916,M,Manual,1,01/08/23 11:44,0.0,United Kingdom


In [ ]:
duplicate_rows = sales_df.duplicated(keep=False)

print("Rows involved in duplicate groups:", duplicate_rows.sum())

print("\nSample duplicate transactions:")
display(
    sales_df.loc[duplicate_rows]
    .sort_values(["CustomerID", "InvoiceNo", "StockCode"])
    .head(20)
)


blank_nps = campaign_df["nps"].astype(str).str.strip().eq("")

print("\nCustomers with blank NPS:")
display(
    campaign_df.loc[blank_nps]
)

print("Number of blank NPS values:", blank_nps.sum())

Rows involved in duplicate groups: 8976

Sample duplicate transactions:


,CustomerID,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
8370,12748,541248,21832,CHOCOLATE CALCULATOR,1,16/01/23 13:04,1.65,United Kingdom
8373,12748,541248,21832,CHOCOLATE CALCULATOR,1,16/01/23 13:04,1.65,United Kingdom
35795,12748,545062,22359,GLASS JAR KINGS CHOICE,1,27/02/23 14:07,2.95,United Kingdom
35798,12748,545062,22359,GLASS JAR KINGS CHOICE,1,27/02/23 14:07,2.95,United Kingdom
35779,12748,545062,22429,ENAMEL MEASURING JUG CREAM,1,27/02/23 14:07,4.25,United Kingdom
35780,12748,545062,22429,ENAMEL MEASURING JUG CREAM,1,27/02/23 14:07,4.25,United Kingdom
35791,12748,545062,22963,JAM JAR WITH GREEN LID,1,27/02/23 14:07,0.85,United Kingdom
35796,12748,545062,22963,JAM JAR WITH GREEN LID,1,27/02/23 14:07,0.85,United Kingdom
56221,12748,547809,20755,BLUE PAISLEY POCKET BOOK,1,25/03/23 13:54,0.85,United Kingdom
56246,12748,547809,20755,BLUE PAISLEY POCKET BOOK,1,25/03/23 13:54,0.85,United Kingdom



Customers with blank NPS:


,CustomerID,response,n_comp,loyalty,nps,n_communications
7,12823,0,3,0,,3
26,12844,1,4,0,,6
787,13952,1,3,0,,5


Number of blank NPS values: 3


In [ ]:
duplicate_group_sizes = (
    sales_df.loc[duplicate_rows]
    .groupby(list(sales_df.columns))
    .size()
)

print("Number of duplicate groups:", len(duplicate_group_sizes))
print("Largest duplicate group:", duplicate_group_sizes.max())

print(
    "Duplicate rows involving cancelled invoices:",
    sales_df.loc[duplicate_rows, "InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

print("\nNPS summary after numeric conversion:")

nps_numeric = pd.to_numeric(
    campaign_df["nps"].str.strip(),
    errors="coerce"
)

print(nps_numeric.describe())
print("Median NPS:", nps_numeric.median())

Number of duplicate groups: 4319
Largest duplicate group: 20
Duplicate rows involving cancelled invoices: 44

NPS summary after numeric conversion:
count    3831.000000
mean        4.528061
std         2.690799
min         0.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        10.000000
Name: nps, dtype: float64
Median NPS: 4.0


In [ ]:
sales_clean = sales_df.copy()
campaign_clean = campaign_df.copy()



sales_clean = sales_clean.drop_duplicates().copy()



sales_clean["InvoiceDate"] = pd.to_datetime(
    sales_clean["InvoiceDate"],
    format="%d/%m/%y %H:%M"
)



sales_clean["is_cancelled"] = (
    sales_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)



sales_clean["is_zero_price"] = (
    sales_clean["UnitPrice"] == 0
)



sales_clean["line_total"] = (
    sales_clean["Quantity"]
    * sales_clean["UnitPrice"]
)



campaign_clean["nps"] = pd.to_numeric(
    campaign_clean["nps"]
    .astype(str)
    .str.strip(),
    errors="coerce"
)

In [11]:
print("Original sales rows:", len(sales_df))
print("Clean sales rows:", len(sales_clean))
print("Rows removed as duplicates:", len(sales_df) - len(sales_clean))

print("\nCancelled rows:", sales_clean["is_cancelled"].sum())
print("Zero-price rows:", sales_clean["is_zero_price"].sum())

print("\nMissing NPS:", campaign_clean["nps"].isna().sum())

print("\nInvoiceDate type:", sales_clean["InvoiceDate"].dtype)
print("NPS type:", campaign_clean["nps"].dtype)

Original sales rows: 337321
Clean sales rows: 332664
Rows removed as duplicates: 4657

Cancelled rows: 6916
Zero-price rows: 23

Missing NPS: 3

InvoiceDate type: datetime64[us]
NPS type: float64


In [12]:
# Validate cleaned datasets

print("SALES CLEAN VALIDATION")
print("Duplicate rows:", sales_clean.duplicated().sum())
print("Missing values:")
print(sales_clean.isna().sum())

print(
    "\nAll cancelled transactions have negative quantity:",
    (sales_clean.loc[sales_clean["is_cancelled"], "Quantity"] < 0).all()
)

print(
    "All non-cancelled transactions have positive quantity:",
    (sales_clean.loc[~sales_clean["is_cancelled"], "Quantity"] > 0).all()
)

print("\nDate range:")
print("First transaction:", sales_clean["InvoiceDate"].min())
print("Last transaction:", sales_clean["InvoiceDate"].max())


print("\nCAMPAIGN CLEAN VALIDATION")
print("Duplicate rows:", campaign_clean.duplicated().sum())
print("Missing NPS:", campaign_clean["nps"].isna().sum())
print("Minimum NPS:", campaign_clean["nps"].min())
print("Maximum NPS:", campaign_clean["nps"].max())

print("\nUnique customers:")
print("Sales:", sales_clean["CustomerID"].nunique())
print("Campaign:", campaign_clean["CustomerID"].nunique())

SALES CLEAN VALIDATION
Duplicate rows: 0
Missing values:
CustomerID       0
InvoiceNo        0
StockCode        0
Description      0
Quantity         0
InvoiceDate      0
UnitPrice        0
Country          0
is_cancelled     0
is_zero_price    0
line_total       0
dtype: int64

All cancelled transactions have negative quantity: True
All non-cancelled transactions have positive quantity: True

Date range:
First transaction: 2023-01-04 10:00:00
Last transaction: 2023-12-09 12:49:00

CAMPAIGN CLEAN VALIDATION
Duplicate rows: 0
Missing NPS: 3
Minimum NPS: 0.0
Maximum NPS: 10.0

Unique customers:
Sales: 3834
Campaign: 3834


In [13]:
# Validate customer IDs across datasets

sales_customers = set(
    sales_clean["CustomerID"].unique()
)

campaign_customers = set(
    campaign_clean["CustomerID"].unique()
)

print(
    "Campaign customers missing from Sales:",
    len(campaign_customers - sales_customers)
)

print(
    "Sales customers missing from Campaign:",
    len(sales_customers - campaign_customers)
)

print(
    "Exact same customer set:",
    sales_customers == campaign_customers
)

print(
    "Duplicate CustomerIDs in Campaign:",
    campaign_clean["CustomerID"].duplicated().sum()
)

Campaign customers missing from Sales: 0
Sales customers missing from Campaign: 0
Exact same customer set: True
Duplicate CustomerIDs in Campaign: 0


In [14]:
# Create customer-level features from transaction data

customer_features = (
    sales_clean
    .groupby("CustomerID")
    .agg(
        total_sales=("line_total", "sum"),
        unique_products=("StockCode", "nunique"),
        number_of_invoices=("InvoiceNo", "nunique"),
        purchase_days=("InvoiceDate", lambda x: x.dt.date.nunique())
    )
    .reset_index()
)

customer_features["average_order_value"] = (
    customer_features["total_sales"]
    / customer_features["number_of_invoices"]
)

display(customer_features.head())

print("Customer features shape:", customer_features.shape)

,CustomerID,total_sales,unique_products,number_of_invoices,purchase_days,average_order_value
0,12346,0.00,1,2,1,0.000000
1,12747,3489.74,39,9,9,387.748889
2,12748,24271.01,1603,187,101,129.791497
3,12749,3868.20,160,8,7,483.525000
4,12820,942.34,55,4,4,235.585000


Customer features shape: (3834, 6)


In [15]:
display(
    sales_clean.loc[
        sales_clean["CustomerID"] == 12346
    ]
)

,CustomerID,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country,is_cancelled,is_zero_price,line_total
9720,12346,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2023-01-18 10:01:00,1.04,United Kingdom,False,False,77183.6
9725,12346,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2023-01-18 10:17:00,1.04,United Kingdom,True,False,-77183.6


In [16]:
# Separate purchases and cancellations

purchase_transactions = sales_clean.loc[
    ~sales_clean["is_cancelled"]
].copy()

cancelled_transactions = sales_clean.loc[
    sales_clean["is_cancelled"]
].copy()


# Purchase behaviour features

purchase_features = (
    purchase_transactions
    .groupby("CustomerID")
    .agg(
        total_sales=("line_total", "sum"),
        unique_products=("StockCode", "nunique"),
        number_of_invoices=("InvoiceNo", "nunique"),
        purchase_days=("InvoiceDate", lambda x: x.dt.date.nunique())
    )
    .reset_index()
)


# Cancellation features

cancellation_features = (
    cancelled_transactions
    .groupby("CustomerID")
    .agg(
        cancelled_invoices=("InvoiceNo", "nunique"),
        cancellation_value=("line_total", lambda x: abs(x.sum()))
    )
    .reset_index()
)


# Net sales including cancellations

net_sales = (
    sales_clean
    .groupby("CustomerID")["line_total"]
    .sum()
    .reset_index(name="net_sales")
)


# Combine all customer features

customer_features = (
    sales_clean[["CustomerID"]]
    .drop_duplicates()
    .merge(purchase_features, on="CustomerID", how="left")
    .merge(cancellation_features, on="CustomerID", how="left")
    .merge(net_sales, on="CustomerID", how="left")
)

customer_features[
    ["cancelled_invoices", "cancellation_value"]
] = customer_features[
    ["cancelled_invoices", "cancellation_value"]
].fillna(0)


customer_features["average_order_value"] = (
    customer_features["total_sales"]
    / customer_features["number_of_invoices"]
)

display(customer_features.head())

print("Customer features shape:", customer_features.shape)

,CustomerID,total_sales,unique_products,number_of_invoices,purchase_days,cancelled_invoices,cancellation_value,net_sales,average_order_value
0,13313,1555.32,47.0,5.0,5.0,0.0,0.00,1555.32,311.064000
1,18097,2479.88,64.0,5.0,4.0,1.0,4.95,2474.93,495.976000
2,16656,8197.04,14.0,14.0,14.0,6.0,57.56,8139.48,585.502857
3,16875,2095.53,93.0,5.0,4.0,4.0,72.06,2023.47,419.106000
4,13094,1703.64,4.0,11.0,11.0,4.0,218.46,1485.18,154.876364


Customer features shape: (3834, 9)


In [17]:
print("Customer 12346:")
display(
    customer_features.loc[
        customer_features["CustomerID"] == 12346
    ]
)

print("\nMissing values in customer features:")
print(customer_features.isna().sum())

print("\nCustomers with no purchase invoices:")
print(customer_features["number_of_invoices"].isna().sum())

print("\nCustomers with cancellations:")
print((customer_features["cancelled_invoices"] > 0).sum())

Customer 12346:


,CustomerID,total_sales,unique_products,number_of_invoices,purchase_days,cancelled_invoices,cancellation_value,net_sales,average_order_value
403,12346,77183.6,1.0,1.0,1.0,1.0,77183.6,0.0,77183.6



Missing values in customer features:
CustomerID              0
total_sales            21
unique_products        21
number_of_invoices     21
purchase_days          21
cancelled_invoices      0
cancellation_value      0
net_sales               0
average_order_value    21
dtype: int64

Customers with no purchase invoices:
21

Customers with cancellations:
1323


In [18]:
# Validate customers with no completed purchases

no_purchase_customers = customer_features.loc[
    customer_features["number_of_invoices"].isna(),
    "CustomerID"
]

print("Customers with no completed purchases:", len(no_purchase_customers))

validation = (
    sales_clean.loc[
        sales_clean["CustomerID"].isin(no_purchase_customers)
    ]
    .groupby("CustomerID")
    .agg(
        total_rows=("InvoiceNo", "size"),
        cancelled_rows=("is_cancelled", "sum"),
        net_sales=("line_total", "sum")
    )
)

validation["all_rows_cancelled"] = (
    validation["total_rows"]
    == validation["cancelled_rows"]
)

display(validation)

print(
    "\nAll 21 customers have only cancelled transactions:",
    validation["all_rows_cancelled"].all()
)

Customers with no completed purchases: 21


,total_rows,cancelled_rows,net_sales,all_rows_cancelled
CustomerID,,,,
12943,1,1,-3.75,True
12967,14,14,-466.15,True
13154,1,1,-611.86,True
13693,4,4,-32.00,True
14437,1,1,-106.40,True
14627,5,5,-21.85,True
14777,1,1,-2.95,True
15363,1,1,-7.95,True
15369,1,1,-1592.49,True



All 21 customers have only cancelled transactions: True


In [19]:
# Fill purchase features for customers with no completed purchases

purchase_columns = [
    "total_sales",
    "unique_products",
    "number_of_invoices",
    "purchase_days",
    "average_order_value"
]

customer_features[purchase_columns] = (
    customer_features[purchase_columns]
    .fillna(0)
)


# Convert count-based features to integers

count_columns = [
    "unique_products",
    "number_of_invoices",
    "purchase_days",
    "cancelled_invoices"
]

customer_features[count_columns] = (
    customer_features[count_columns]
    .astype(int)
)


# Create cancellation rate

customer_features["total_invoice_activity"] = (
    customer_features["number_of_invoices"]
    + customer_features["cancelled_invoices"]
)

customer_features["cancellation_rate"] = (
    customer_features["cancelled_invoices"]
    / customer_features["total_invoice_activity"]
)

In [20]:
print("Missing values:")
print(customer_features.isna().sum())

print(
    "\nCustomers with cancellation rate = 100%:",
    (customer_features["cancellation_rate"] == 1).sum()
)

print("\nShape:", customer_features.shape)

display(
    customer_features.loc[
        customer_features["CustomerID"].isin(
            [12346, 12943, 12747]
        )
    ]
)

Missing values:
CustomerID                0
total_sales               0
unique_products           0
number_of_invoices        0
purchase_days             0
cancelled_invoices        0
cancellation_value        0
net_sales                 0
average_order_value       0
total_invoice_activity    0
cancellation_rate         0
dtype: int64

Customers with cancellation rate = 100%: 21

Shape: (3834, 11)


,CustomerID,total_sales,unique_products,number_of_invoices,purchase_days,cancelled_invoices,cancellation_value,net_sales,average_order_value,total_invoice_activity,cancellation_rate
403,12346,77183.60,1,1,1,1,77183.60,0.00,77183.600000,2,0.5
446,12747,3489.74,39,9,9,0,0.00,3489.74,387.748889,9,0.0
875,12943,0.00,0,0,0,1,3.75,-3.75,0.000000,1,1.0


In [21]:

# Merge campaign data with customer-level transaction features

master_df = campaign_clean.merge(
    customer_features,
    on="CustomerID",
    how="left",
    validate="one_to_one"
)

print("Master dataset shape:", master_df.shape)

print("\nMissing values:")
print(master_df.isna().sum())

print("\nDuplicate CustomerIDs:")
print(master_df["CustomerID"].duplicated().sum())

display(master_df.head())

Master dataset shape: (3834, 16)

Missing values:
CustomerID                0
response                  0
n_comp                    0
loyalty                   0
nps                       3
n_communications          0
total_sales               0
unique_products           0
number_of_invoices        0
purchase_days             0
cancelled_invoices        0
cancellation_value        0
net_sales                 0
average_order_value       0
total_invoice_activity    0
cancellation_rate         0
dtype: int64

Duplicate CustomerIDs:
0


,CustomerID,response,n_comp,loyalty,nps,n_communications,total_sales,unique_products,number_of_invoices,purchase_days,cancelled_invoices,cancellation_value,net_sales,average_order_value,total_invoice_activity,cancellation_rate
0,12346,1,2,1,7.0,8,77183.60,1,1,1,1,77183.60,0.00,77183.600000,2,0.500000
1,12747,0,1,1,3.0,3,3489.74,39,9,9,0,0.00,3489.74,387.748889,9,0.000000
2,12748,1,0,1,9.0,6,28868.19,1602,175,100,12,4597.18,24271.01,164.961086,187,0.064171
3,12749,0,4,1,2.0,5,4090.88,160,5,4,3,222.68,3868.20,818.176000,8,0.375000
4,12820,0,4,1,2.0,2,942.34,55,4,4,0,0.00,942.34,235.585000,4,0.000000


In [22]:
# Overall campaign response rate

response_rate = master_df["response"].mean() * 100

print("Total customers:", len(master_df))
print("Customers who responded:", master_df["response"].sum())
print("Overall response rate:", round(response_rate, 2), "%")

Total customers: 3834
Customers who responded: 1555
Overall response rate: 40.56 %


In [23]:
# Overall campaign response rate

response_rate = master_df["response"].mean() * 100

print("Total customers:", len(master_df))
print("Customers who responded:", master_df["response"].sum())
print("Overall response rate:", round(response_rate, 2), "%")

Total customers: 3834
Customers who responded: 1555
Overall response rate: 40.56 %


In [24]:
# Create NPS categories

master_df["nps_category"] = pd.cut(
    master_df["nps"],
    bins=[-1, 6, 8, 10],
    labels=["Detractor", "Passive", "Promoter"]
)

print(master_df["nps_category"].value_counts(dropna=False))

display(
    master_df[
        ["CustomerID", "nps", "nps_category"]
    ].head(10)
)

nps_category
Detractor    2794
Passive       776
Promoter      261
NaN             3
Name: count, dtype: int64


,CustomerID,nps,nps_category
0,12346,7.0,Passive
1,12747,3.0,Detractor
2,12748,9.0,Promoter
3,12749,2.0,Detractor
4,12820,2.0,Detractor
5,12821,5.0,Detractor
6,12822,6.0,Detractor
7,12823,NaN,NaN
8,12824,2.0,Detractor
9,12826,8.0,Passive


In [25]:
# Inspect numerical variables before deciding on recoding

numeric_columns = [
    "n_comp",
    "n_communications",
    "nps",
    "total_sales",
    "unique_products",
    "number_of_invoices",
    "purchase_days",
    "cancelled_invoices",
    "cancellation_value",
    "net_sales",
    "average_order_value",
    "total_invoice_activity",
    "cancellation_rate"
]

summary = master_df[numeric_columns].describe().T

summary["unique_values"] = (
    master_df[numeric_columns]
    .nunique()
)

display(summary)

,count,mean,std,min,25%,50%,75%,max,unique_values
n_comp,3834.0,2.577986,1.483994,0.00,1.00000,3.00,4.0000,5.00,6
n_communications,3834.0,5.335159,2.074677,2.00,4.00000,5.00,7.0000,9.00,8
nps,3831.0,4.528061,2.690799,0.00,2.00000,4.00,7.0000,10.00,11
total_sales,3834.0,1770.490510,7106.967646,0.00,287.45500,631.23,1527.0100,231822.69,3747
unique_products,3834.0,58.526604,78.572942,0.00,15.00000,34.00,74.0000,1602.00,314
number_of_invoices,3834.0,4.005216,6.578192,0.00,1.00000,2.00,4.0000,175.00,51
purchase_days,3834.0,3.647887,5.325479,0.00,1.00000,2.00,4.0000,106.00,47
cancelled_invoices,3834.0,0.763172,1.912989,0.00,0.00000,0.00,1.0000,43.00,23
cancellation_value,3834.0,136.427994,3129.911197,0.00,0.00000,0.00,13.5000,168469.60,985
net_sales,3834.0,1634.062515,6151.306036,-4287.63,275.56500,613.98,1489.0700,228603.88,3771


In [26]:
# Create categorical cancellation indicator

master_df["has_cancellation"] = (
    master_df["cancelled_invoices"] > 0
).astype(int)

print(
    master_df["has_cancellation"]
    .value_counts()
    .sort_index()
)

print("\nCancellation rate by category:")

display(
    master_df.groupby("has_cancellation")[
        "cancellation_rate"
    ].describe()
)

has_cancellation
0    2511
1    1323
Name: count, dtype: int64

Cancellation rate by category:


,count,mean,std,min,25%,50%,75%,max
has_cancellation,,,,,,,,
0,2511.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
1,1323.0,0.306279,0.169307,0.012195,0.186607,0.266667,0.4,1.0


In [27]:
# Final validation of the master dataset

print("MASTER DATASET FINAL VALIDATION")

print("\nShape:", master_df.shape)
print("Unique customers:", master_df["CustomerID"].nunique())
print("Duplicate CustomerIDs:", master_df["CustomerID"].duplicated().sum())

print("\nMissing values:")
print(master_df.isna().sum())

print("\nResponse values:", sorted(master_df["response"].unique()))
print("Loyalty values:", sorted(master_df["loyalty"].unique()))

print(
    "NPS within valid range:",
    master_df["nps"].dropna().between(0, 10).all()
)

print(
    "Cancellation rate within 0-1:",
    master_df["cancellation_rate"].between(0, 1).all()
)

print(
    "Purchase days <= number of invoices:",
    (
        master_df["purchase_days"]
        <= master_df["number_of_invoices"]
    ).all()
)

print(
    "Net sales <= total sales:",
    (
        master_df["net_sales"]
        <= master_df["total_sales"]
    ).all()
)

print(
    "Cancellation indicator consistent:",
    (
        master_df["has_cancellation"]
        == (master_df["cancelled_invoices"] > 0).astype(int)
    ).all()
)

MASTER DATASET FINAL VALIDATION

Shape: (3834, 18)
Unique customers: 3834
Duplicate CustomerIDs: 0

Missing values:
CustomerID                0
response                  0
n_comp                    0
loyalty                   0
nps                       3
n_communications          0
total_sales               0
unique_products           0
number_of_invoices        0
purchase_days             0
cancelled_invoices        0
cancellation_value        0
net_sales                 0
average_order_value       0
total_invoice_activity    0
cancellation_rate         0
nps_category              3
has_cancellation          0
dtype: int64

Response values: [np.int64(0), np.int64(1)]
Loyalty values: [np.int64(0), np.int64(1)]
NPS within valid range: True
Cancellation rate within 0-1: True
Purchase days <= number of invoices: True
Net sales <= total sales: True
Cancellation indicator consistent: True


In [28]:
# Save Phase 1 outputs

INTERIM_DATA = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

sales_clean.to_csv(
    INTERIM_DATA / "sales_clean.csv",
    index=False
)

campaign_clean.to_csv(
    INTERIM_DATA / "campaign_clean.csv",
    index=False
)

customer_features.to_csv(
    PROCESSED_DATA / "customer_features.csv",
    index=False
)

master_df.to_csv(
    PROCESSED_DATA / "master_dataset.csv",
    index=False
)

print("Phase 1 datasets saved successfully.")

Phase 1 datasets saved successfully.


In [29]:
for file in INTERIM_DATA.glob("*.csv"):
    print("INTERIM:", file.name, "-", round(file.stat().st_size / 1_000_000, 2), "MB")

for file in PROCESSED_DATA.glob("*.csv"):
    print("PROCESSED:", file.name, "-", round(file.stat().st_size / 1_000_000, 2), "MB")

INTERIM: campaign_clean.csv - 0.07 MB
INTERIM: sales_clean.csv - 36.46 MB
PROCESSED: customer_features.csv - 0.22 MB
PROCESSED: master_dataset.csv - 0.31 MB
